In [3]:
import json
import math
import random
from functools import reduce, wraps

In [4]:
with open('./data/walk_22-guard-else-is-not-what-you-think.json', 'r') as file:
    # data = json.loads(file)
    contents = file.read()
    walk_22 = json.loads(contents)
coords = walk_22['features'][0]['geometry']['coordinates']
p1 = { "longitude": coords[0][0], "latitude": coords[0][1] }
p2 = { "longitude": coords[1][0], "latitude": coords[1][1] }
p3 = { "longitude": coords[2][0], "latitude": coords[2][1] }
p4 = { "longitude": coords[3][0], "latitude": coords[3][1] }
p5 = { "longitude": coords[4][0], "latitude": coords[4][1] }



In [5]:
with open('./data/walk_31-i-m-on-unifi.json', 'r') as file:
    contents = file.read()
    walk_31 = json.loads(contents)


In [13]:
# Some constants are defined for use in the entire module.

# Terrain coeffcients to characterize walking surface.
TERRAIN_COEFFCIENTS = {
    "BLACKTOP": 1.0, # Paved road / treadmill
    "DIRT": 1.1,     # Dirt path, packed trail
    "LIGHT": 1.2,    # Light off-trail, grass
    "SOFT": 1.5,     # Soft sand, deep grass, loose gravel
    "HEAVY": 1.8     # Snow, heavy brush, swamp
}

# Defaults for smoothing out jittery GPS elevation data.
SMOOTH_DEFAULT = True
SMOOTH_DEFAULT_WINDOW = 5

# Maximum plausible walking speed (m/s).
# Speeds higher than this are clamped to this value.
MAX_SPEED_MS = 4.0

# Minimum segment distance. Filters out GPS jitter.
MIN_SEGMENT_DIST_M = 0.5

# Conversion: 1Kcal = 4184 joules
JOULES_PER_KCAL = 4184

# Minimum Mechanics derived constants:
# Table 4, Ludlow & Weyland 2017
MM_COEFFICIENTS = {
    "C1": 0.32,              # grade influence on minimum walking metabolic rate
    "C2": 0.19,              # grade influence on speed-dependent walking metabolic rate
    "C3": 2.66,              # velocity squared coefficient
    "VO2_WALK_MIN": 3.28,    # ml O2 kg-total^-1 min^-1, minimum walking metabolic rate
    "C_DECLINE": 0.73        # fraction of level-grade walking cost applied in decline
}


# Mean measured supine resting metablic rate across all 32 study subjects (ml O2 kg-body^-1 min^-1).
# Used as the default VO2-rest term if no subject-specific resting metabloic rate is given.
# Ludlow & Weyland 2017
DEFAULT_RESTING_VO2 = 3.05

# Standard caloric equivalent of oxygen: ~5kcal per liter O2 per 1000ml.  Expressed here per ml
# for direct multiplication against VO2 rates in ml O2 min^-1.
KCAL_PER_ML_O2 = 0.005

In [35]:
# Debug printing or not.
DEBUG_PRINTING = True

In [36]:
# Decorator to wrap around print() to suppress output if DEBUG_PRINTING == False
def print_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        if DEBUG_PRINTING:
            return func(*args, **kwargs)
        return None
    return wrapper


print = print_decorator(print)
# test if this works
# print("Does this print?")

In [16]:
# Convert a number of milliseconds to seconds.
def m2s(milliseconds: int) -> int:
    """Convert a number of milliseconds to seconds.

    Args:
        milliseconds (int): Time in milliseconds.

    Returns:
        int: Time converted into seconds.
    """
    # print(f"milliseconds type: {type(milliseconds)}")
    seconds = int(milliseconds / 1000)
    # print(f"seconds {seconds}, type: {type(seconds)}")
    return seconds


i = random.randrange(0, len(coords))
print(f"random index: {i}")
point1 = coords[i]
point2 = coords[i + 100]
print(m2s(point2[5] - point1[5]))

random index: 1582
100


In [17]:
# Convert a number of milliseconds to minutes.
def m2m(milliseconds: int) -> float:
    """Convert a number of milliseconds to minutes.

    Args:
        milliseconds (int): Time in milliseconds.

    Returns:
        float: Time converted into minutes.
    """
    # print(f"milliseconds type: {type(milliseconds)}")
    # print(f"milliseconds -> seconds: {m2s(milliseconds)}")
    minutes = milliseconds / 60000
    # print(f"minutes type: {type(minutes)}")
    return minutes


# i = random.randrange(0, len(coords))
print(f"random index: {i}")
point1 = coords[i]
point2 = coords[i + 100]
print(m2m(point2[5] - point1[5]))

random index: 1582
1.6666666666666667


In [18]:
# Convert compass degress to radians.
def rads(degrees: float) -> float:
    """Convert compass degrees to radians.

    Args:
        degrees (float): Compass degress value.

    Returns:
        float: The calculated radians value.
    """
    return degrees * (math.pi / 180)


i = random.randrange(0, len(coords))
for j in range(i, i + 10):
    print(f"heading degrees: {coords[j][2]} radians: {rads(coords[j][2])}")


heading degrees: 135.2740020751953 radians: 2.3609767285618
heading degrees: 135.2740020751953 radians: 2.3609767285618
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 131.3022918701172 radians: 2.2916573085481278
heading degrees: 106.46765907583986 radians: 1.8582111977642288


In [19]:
# Convert radians to compass degress.
def degs(radians: float) -> float:
    """Convert radians to compass degrees.

    Args:
        radians (float): Radians value

    Returns:
        float: The calculated degrees value.
    """
    return radians * (180 / math.pi)


i = random.randrange(0, len(coords))
for j in range(i, i + 10):
    print(f"heading degrees: {coords[j][2]}, from radians: {degs(rads(coords[j][2]))}")


heading degrees: 255.87157281742415, from radians: 255.87157281742415
heading degrees: 252.60438928842703, from radians: 252.60438928842703
heading degrees: 250.14690846291074, from radians: 250.14690846291074
heading degrees: 264.35257358480203, from radians: 264.35257358480203
heading degrees: 262.54861628069176, from radians: 262.54861628069176
heading degrees: 261.4204145609704, from radians: 261.4204145609704
heading degrees: 253.9169913270976, from radians: 253.91699132709763
heading degrees: 258.0925127922811, from radians: 258.0925127922811
heading degrees: 255.45803437499757, from radians: 255.45803437499757
heading degrees: 262.50597749617714, from radians: 262.50597749617714


In [20]:
def pointDistance(p1: dict, p2: dict, u = "metric") -> float:
    """Calculate the Haversine distance between two GPS points.

    Args:
        p1 (dict): Dictionary containing latitude and longitude values.
        p2 (dict): Dictionary containing latitude and longitude values.
        u (string): String value indicating unit system to use.

    Returns:
        float: The Haversine distance between GPS points p1 and p2.

    Raises:
        ValueError: if p1 argument is missing longitude or latitude values.
        ValueError: if p2 argument is missing longitude or latitude values.
    """
    if "longitude" not in p1 or "latitude" not in p1:
        raise ValueError(f"Point p1 argument is requires longitude and latitude values.")
    if "longitude" not in p2 or "latitude" not in p2:
        raise ValueError(f"Point p2 argument is requires longitude and latitude values.")
    earthRadiusKm = 6371
    earthRadiusMeters = 6371000
    earthRadiusMiles = 3959
    _u = u.lower()
    r = None
    if _u == 'm' or _u == 'meters':
        r = earthRadiusMeters
    elif _u == 'km' or _u == 'kilometers':
        r = earthRadiusKm
    elif _u == 'mi' or _u == 'miles' or _u == 'imperial':
        r = earthRadiusMiles
    else:
        r = earthRadiusMeters
    #print(r, _u)
    dLat = rads(p2['latitude'] - p1['latitude'])
    dLon = rads(p2['longitude'] - p1['longitude'])
    lat1 = rads(p1['latitude'])
    lat2 = rads(p2['latitude'])
    a = math.sin(dLat / 2) * math.sin(dLat / 2) \
        + math.sin(dLon / 2) * math.sin(dLon / 2) * math.cos(lat1) * math.cos(lat2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return c * r


print(pointDistance(p1, p2, u ='mi'))
print(pointDistance(p1, p3, u ='mi'))
print(pointDistance(p1, p4, u ='mi'))
print(pointDistance(p1, p5, u ='mi'))
# p6 = { "latitude": 10 }
# print(pointDistance(p1, p6, u ='mi'))
# help(pointDistance)

0.00010394011574007407
0.00021639922403606193
0.0003388536049015972
0.0026016707988299878


In [21]:
# Calculate the difference in altitude between two points.
def calculateVerticalInterval(alt1: float, alt2: float) -> float:
    """Calculate the difference in altitude between two points.

    Args:
        alt1 (float): First altitude value.
        alt2 (float): Second altitude value.

    Returns:
        float: Altitude difference.
    """
    return alt2 - alt1


r = random.randrange(0, len(coords))
print(f"random index: {r}")
point1 = coords[r]
point2 = coords[r+100]
print(f"vertical distance between point1 & point2: {calculateVerticalInterval(point1[3], point2[3])}")

random index: 6530
vertical distance between point1 & point2: 8.116248959675431


In [22]:
# Calculate the slope between two points.
def calculateSlopeGrade(point1: dict, point2: dict) -> dict:
    """Calcuate the slope between two GPS points.

    Args:
        point1 (dict): Dictionary with longitude and latitude properties.
        point2 (dict): Dictionary with longitude and latitude properties.

    Returns:
        dict: Dictionary with grade and angleDegrees properties.
    """
    horizontalDistance = pointDistance(point1, point2)
    verticalDistance = calculateVerticalInterval(point1["altitude"], point2["altitude"])
    if horizontalDistance == 0:
        return { "grade": math.inf, "angleDegrees": 90 }
    slope = verticalDistance / horizontalDistance
    grade = slope * 100
    angle = math.atan(slope) * 180
    return {
        "grade": grade,
        "angleDegrees": angle / math.pi
    }


r = random.randrange(0, len(coords))
print(f"random index: {r}")
point1 = { "longitude": coords[r][0], "latitude": coords[r][1], "altitude": coords[r][3] }
point2 = { "longitude": coords[r + 1][0], "latitude": coords[r + 1][1], "altitude": coords[r + 1][3] }
print(f"hill grade between point1 & point2: {calculateSlopeGrade(point1, point2)}")

random index: 389
hill grade between point1 & point2: {'grade': 25.68289483750934, 'angleDegrees': 14.40390169881726}


In [23]:
# Apply a simple rolling-average smoother to the altitude values in a coordinates array.
def smoothAltitude(coords: List[List[float]], windowSize: int = SMOOTH_DEFAULT_WINDOW) -> List[List[float]]:
    """Apply a simple rolling-average smoothing function to the altitude values in a coordinates array.
        Raw GPS altitude can have +-5 to 15 m of noise, which can create artificial grade spikes that inflate calorie estimates.

    Args:
        coords (List[List[float]]: List of coordinate arrays.
        windowSize (int): Number of points to average (odd number recommended).

    Returns:
        List[List[float]]: New coordinates array with smoothed altitudes.
    """
    half = math.floor(windowSize / 2)
    # print(f"windowSize: {windowSize}, half: {half}")
    smoothed = list()
    i = 0
    n = len(coords)
    # print(f"coords length: {n}")
    while i < n:
        # print(f"\tstarting loop: {i}")
        start = max(0, i - half)
        end = min(n - 1, (i + half) + 2) # + 2 because slice end index is non-inclusive
        # print(f"\tstart: {start}, end: {end}")
        slice = coords[start:end]
        # print(f"\tslice (length {len(slice)}): {slice}\n")
        validAlts = [x[3] for x in slice if x[3] is not None]
        # print(f"\tvalidAlts: {validAlts}")
        averageAltitude = reduce(lambda acc, curr: acc + curr, validAlts, 0) / len(validAlts) if len(validAlts) > 0 else slice[3]
        smoothed.append([coords[i][0], coords[i][1], coords[i][2], averageAltitude, coords[i][4], coords[i][5]])
        # print(f"\tsmoothed altitude: {averageAltitude}, {validAlts}\n")
        i = i + 1
    return smoothed


# coordinateSlice = coords[0:25]
# print(coordinateSlice)
# print(smoothAltitude(coordinateSlice))

In [24]:
# Simple MET based calorie estimate.
def simpleCalories(minutes: int = 1, weights: dict = { "body": 0, "ruck": 0, "water": 0 }, MET: float = 7.5) -> float:
    """The simplest calorie estimating function.  Calculates the ratio of energy spent per unit time during a specific
    physical activity to a reference value of 3.5 ml O2 / (kg·min).

    Args:
        minutes (float): Time spent expending energy, in minutes.
        weights (dict): Collection of weight values, in kilograms.
        MET (float): The Metabolic Equivalent Task number of activity.

    Raises:
        ValueError: If minutes is not a valid, positive number.
        ValueError: If weights.body is not a valid, positive number.
        ValueError: If MET is not a valid, positive number.
        
    Returns:
        float: Number of calories burned.
    """
    if minutes <= 0 or minutes is None:
        raise ValueError(f"Minutes must be a positive, finite number. (Supplied {minutes})")
    if weights["body"] <= 0 or weights["body"] is None:
        raise ValueError(f"Body weight must be a positive, finite number.  (Supplied {weights["body"]}")
    if MET <= 0 or MET is None:
        raise ValueError(f"MET must be a positive, finite number.  (Supplied {MET})")
    COMBINED = weights["body"] + weights["ruck"] + weights["water"]
    # print(COMBINED)
    return ((MET * 3.5 * COMBINED) / 200) * minutes
    

# simpleCalories(0, {"body": 10}, MET = 7.5)
# simpleCalories(10, {"body": 0}, MET = 7.5)
# simpleCalories(10, {"body": 60}, MET = -1)
print(simpleCalories(10, {"body": 60, "ruck": 5, "water": 1}, 7.5))

86.625


In [25]:
# Corrective factor for downhill (G < 0) segments of the hike.
def santeeCorrective(W: float, L: float, V: float, G: float, n: float) -> float:
    """Corrective factor for downhill (G < 0) segments of the hike.

    Args:
        W (float): Body weight measured in kg.
        L (float): Load weight measured in kg.
        V (float): Walking speed in m/s.
        G (float): Hill grade as a percentage (e.g 10 for 10% incline, -5 for decline).
        n (float): Terrain characterization coefficient.

    Returns:
        float: Downhill corrective factor in Watts.
    """
    return n * ( \
        (G * (W + L) * V) / 3.5 \
        - ((W + L) * (((G + 6) ** 2) / W)) \
        + (25 * (V ** 2)) \
    )



In [26]:
# Calculate the metabolic rate (Watts) using Pandolf-Santee predictive model.
def pandolfMetabolicRate(W: float, L: float, V: float, G: float, n: float) -> float:
    """Calculate the metabolic rate (Watts) using Pandolf-Santee predictive model.

    Args:
        W (float): Body weight measured in kg.
        L (float): Load weight measured in kg.
        V (float): Walking speed in m/s.
        G (float): Hill grade as a percentage (e.g 10 for 10% incline, -5 for decline).
        n (float): Terrain characterization coefficient.

    Returns:
        float: Metabolic rate in Watts (should always be >= 0).
    """
    if V <= 0:
        return 0
    loadRatio = L / W
    M = 1.5 * W \
        + 2 * (W + L) * loadRatio ** 2 \
        + n * (W + L) * (1.5 * V ** 2 + 0.35 * V  * G)
    correction = 0
    if G < 0:
        correction = santeeCorrective(W, L, V, G, n)
    # the equation can return negative values on steep descents so clamp to 0.
    return max(0, M - correction)



In [27]:
# Processes a single segment (two consecutive GPS points and returns metabolic and distance data.
def processPandolfSegment(point1: List, point2: List, W: float, L: float, H2O: float, n: float) -> dict | None:
    """Processes a single segment (two consecutive GPS points and returns metabolic and distance data.

    Args:
        point1 (List): [longitude, latiude, heading, altitude, accuracy, timestamp]
        point1 (List): [longitude, latiude, heading, altitude, accuracy, timestamp]
        W (float): Body weight measured in kg.
        L (float): Load weight carried measured in kg.
        H20 (float): Water weight carried measured in kg.
        n (float): Terrain characterization coefficient.

    Returns:
        dict | None: Segment result or None if the segment should be skipped.
    """
    lon1, lat1, _, alt1, _, t1 = point1
    lon2, lat2, _, alt2, _, t2 = point2
    p1 = { "longitude": lon1, "latitude": lat1, "altitude": alt1 }
    p2 = { "longitude": lon2, "latitude": lat2, "altitude": alt2 }
    horizontalDistance = pointDistance(p1, p2)
    durationSec = m2s(t2 - t1)
    # skip GPS jitter, stationary points, or out-of-order timestamps
    if durationSec <= 0 or horizontalDistance < MIN_SEGMENT_DIST_M:
        return None
    slopeGrade = calculateSlopeGrade(p1, p2)
    grade = slopeGrade["grade"]
    altitudeDiff = alt2 = alt1
    # Derived speed, clamped to MAX_SPEED_MS to guard against GPS outliers.
    speed = min(horizontalDistance / durationSec, MAX_SPEED_MS)
    # Metabolic rate (Watts) for this segment.
    metabolicRateWatts = pandolfMetabolicRate(W, L + H2O, speed, grade, n)
    # Energy expended = power * time (joules), converted to kcal.
    kcal = (metabolicRateWatts * durationSec) / JOULES_PER_KCAL
    return {
        "horizontalDistance": horizontalDistance,
        "altitudeDiff": altitudeDiff,
        "grade": grade,
        "speed": speed,
        "durationSec": durationSec,
        "metabolicRateWatts": metabolicRateWatts,
        "kcal": kcal
    }



In [33]:
# Use the Pandolf-Santee predictive model to calculate the total (and per-segment) calorie expenditure for a GPS track.
def pandolfCalories(coords: List[List[float]] = [], options: dict = {}) -> dict:
    """Use the Pandolf-Santee predictive model to calculate the total (and per-segment) calorie expenditure for a GPS track.

    Args:
        coords (List[List[float]]): GPS coordinates array.  Each element:
            [longitude, latitude, heading, altitude (m), accuracy (m), timestamp (ms)]
        options (dict): Options
        options["bodyWeightKg"] (float): Body weight in kg (required).
        options["loadKg"] = 0 (float): Load/pack weight in kg (optional).
        options["waterKg"] = 0 (float): Water weight in kg carried (optional).
        options["terrain"] = 1.1 (float): Terrain coefficient (optional).  Use TERRAIN_COEFFICIENTS.
        options["smooth"] = True (Boolean): Whether to smooth GPS altitude values (optional).
        options["smoothWindow"] = 5 (int): Rolling average window size for smoothing (optional).
        options["returnSegments"] = False (Boolean): Return array of all segments calculated (optional)?

    Raises:
        ValueError: If coords array contains less than 2 items.
        ValueError: If required body weight is < 0, null, or otherwise invalid.

    Returns:
        dict: Results
        {
            totalKcal,        # Total calories burned.
            totalDistanceM,   # Total horizontal distance (meters).
            totalDurationSec, # Total elapsed time (seconds).
            avgSpeedMs,       # Average speed (m/s).
        }
    """
    bodyWeightKg = options.get("bodyWeightKg", 0)
    loadKg = options.get("loadKg", 0)
    waterKg = options.get("waterKg", 0)
    terrain = options.get("terrain", 1.1)
    smooth = options.get("smooth", True)
    smoothWindow = options.get("smoothWindow", SMOOTH_DEFAULT_WINDOW)
    returnSegments = options.get("returnSegments", False)
    if len(coords) < 2:
        raise ValueError(f"The coordinates array needs at least 2 elements, {len(coords)} provided.")
    if not bodyWeightKg or bodyWeightKg <= 0:
        raise ValueError(f"options.bodyWeightkg is required and must be a positive number, {bodyWeightKg} provided.")
    track = smoothAltitude(coords, smoothWindow) if smooth else coords
    print(len(track))
    segments = []
    totalKcal = 0
    totalDistanceM = 0
    totalDurationSec = 0
    for i in range(0, len(track)):
        seg = processPandolfSegment(track[i - 1], track[i], bodyWeightKg, loadKg, waterKg, terrain)
        if seg:
            totalKcal += seg["kcal"]
            totalDistanceM += seg["horizontalDistance"]
            totalDurationSec += seg["durationSec"]
            segments.append(seg)

    avgSpeedMs = (totalDistanceM / totalDurationSec) if (totalDurationSec > 0) else 0
    results = {
        "totalKcal": totalKcal,
        "totalDistanceM": totalDistanceM,
        "totalDurationSec": totalDurationSec,
        "avgSpeedMs": avgSpeedMs
    }
    if returnSegments:
        results["segments"] = segments
    return results



In [34]:
opts = {
    "bodyWeightKg": walk_22["features"][0]["properties"]["weights"]["body"],
    "ruckWeightKg": walk_22["features"][0]["properties"]["weights"]["ruck"],
    "smooth": True,
    "smoothWindow": 5
}
coo = coords[:1000]
pandolfCalories(coords, opts)

7359


{'totalKcal': 1320.360406608483,
 'totalDistanceM': 6090.789713205702,
 'totalDurationSec': 3953,
 'avgSpeedMs': 1.5408018500393883}

In [30]:
opts = {
    "bodyWeightKg": walk_31["features"][0]["properties"]["weights"]["body"],
    "ruckWeightKg": walk_31["features"][0]["properties"]["weights"]["ruck"],
    "smooth": True,
    "smoothWindow": 5
}
pandolfCalories(walk_31["features"][0]["geometry"]["coordinates"], opts)

{'totalKcal': 1291.057910387771,
 'totalDistanceM': 6214.008033363585,
 'totalDurationSec': 3956,
 'avgSpeedMs': 1.5707805948846272}

In [37]:
# Calculate the resting metabolic rate based on inputs provided.
def mResting(height: float, weight: float, age: int, sex: string) -> float:
    """Calculate the resting metabolic rate based on the inputs provided.

    Args:
        height (float): Body height, measured in cm.
        weight (float): Body weight, measured in kg.
        age (int): Age, in years.
        sex (string) = 'm'|'f': Male of female.

    Returns:
        float: Resting metabolic rate in Watts per kg.
    """
    s = 5 if sex == 'm' else -161
    kcals = (10 * weight) + (6.25 * height) - (5 - age) + s
    joules = kcals * JOULES_PER_KCAL
    watts = joules / 86400
    return watts / weight

print(mResting(162, 165/2.2, 53, 'm'))

1.1722302469135801


In [38]:
# Calculate metabolic rate (W·kg^-1) using the LCDA predictive model.
def lcdaMetabolicRate(L_Bp: float, S: float, G: float, n: float, rM: dict) -> float:
    """Calculate metabolic rate (W·kg^-1) using the LCDA predictive model.

    Implements equation 4 from Looney et al. (2022), which combines the
    level-walking LCDA backpacking equation (eq. 2) with the LCDA-graded
    walking equation (eq. 3) and terrain coefficient.

    Args:
        L_Bp (float): Backpack load divided by body mass (dimensionless ratio,
                      e.g. 0.18 for a load equal to 18% of body mass).
        S (float): Walking speed, in m/s.
        G (float): Grade as decimal (rise/run, e.g. .05 for 5% incline, -.05 for decline).
        n (float): Terrain coefficient (η).
        rM (dict): Values for calculating resting metabolic rate.
        rM["height"] (float) Body height in cm.
        rM["weight"] (float): Body weight in kg.
        rM["age"] (int): Age, in years.
        rM["sex"] = 'm'|'f' (string): Male or female.

    Returns:
        float: Body-mass-specific metabolic rate in Watts per kg (>= 0).
    """
    # Eq. 3 — LCDA-graded walking term (W·kg^-1); G is decimal grade (rise/run).
    def M_grade(s, g):
        return 34 * s * g * (1 - 1.05 ** (1 - 1.1 ** (100 * g + 32)))

    if S <= 0:
        return 0

    M_resting = mResting(rM["height"], rM["weight"], rM["age"], rM["sex"])
    speedTerms = 1.78 * S ** 0.58 + 0.27 * S ** 4
    gradeTerms = M_grade(S, G)
    loadFactor = 1 + 1.96 * L_Bp ** 1.36

    # Eq. 4 — combined LCDA backpacking + graded + terrain equation (W·kg^-1)
    return max(0, M_resting + (0.19 + n * (speedTerms + gradeTerms)) * loadFactor)



In [52]:
# Process a single segment (two consecutive GPS points) and return metabolic 
# and distance data for that segment.
def processLcdaSegment(point1: List, point2: List, W: float, L: float, H2O: float, n: float, rM: dict) -> dict | None:
    """Process a single segment (two consecutive GPS points) and return metabolic and
    distance data for that segment.
    
    Args:
        point1 (List): [longitude, latitude, heading, altitude, accuracy, timestamp]
        point2 (List): [longitude, latitude, heading, altitude, accuracy, timestamp]
        W (float): Body weight in kg.
        L (float): Load carried in kg (pack, excluding water).
        H2O (float): Water carried in kg.
        n (float): Terrain coefficient (η).
        rM (dict): Values for calculating resting metabolic rate.
        rM["height"] - Body height in cm.
        rM["weight"] - Body weight in kg.
        rM["age"] - Age, in years.
        rM["sex"] = 'm'|'f' (string): Male or female.

    Returns:
        dict | None: Segment result or None if the segment should be skipped.
    """
    lon1, lat1, _, alt1, _, t1 = point1
    lon2, lat2, _, alt2, _, t2 = point2

    p1 = { "longitude": lon1, "latitude": lat1, "altitude": alt1 }
    p2 = { "longitude": lon2, "latitude": lat2, "altitude": alt2 }
    horizontalDistance = pointDistance(p1, p2)
    durationSec = m2s(t2 - t1) # seconds

    # Skip GPS jitter, stationary points, or out-of-order timestamps.
    if (durationSec <= 0 or horizontalDistance < MIN_SEGMENT_DIST_M):
        return None

    # Find the elevation change as slope between two points.
    slopeGrade = calculateSlopeGrade(p1, p2)
    grade = slopeGrade["grade"]
    # Uses horizontal distance as the run (standard for hiking/trail grade).
    altitudeDiff = alt2 - alt1

    # Derived speed - clamped to MAX_SPEED_MS to guard against GPS outliers.
    speed = min(horizontalDistance / durationSec, MAX_SPEED_MS)

    # LCDA equation uses L_Bp = load/body_mass (dimensionless).
    L_Bp = (L + H2O) / W
    # LCDA equation uses grade as decimal (not %).
    decimalGrade = grade / 100

    # lcdaMetabolicRate returns Watts per kg; multiply by body mass to get total Watts.
    metabolicRatePerKg = lcdaMetabolicRate(L_Bp, speed, decimalGrade, n, rM)
    lcdaMetabolicRateWatts = metabolicRatePerKg * W

    # Energy expended = power × time (joules), converted to kcal.
    kcal = (lcdaMetabolicRateWatts * durationSec) / JOULES_PER_KCAL

    return {
        "horizontalDistance": horizontalDistance,         # meters
        "altitudeDiff": altitudeDiff,                     # meters
        "grade": grade,                                   # percentage
        "speed": speed,                                   # m/s
        "durationSec": durationSec,                       # seconds
        "lcdaMetabolicRateWatts": lcdaMetabolicRateWatts, # Watts
        "kcal": kcal                                      # kilocalories
      }

In [59]:
# Use the LCDA predictive model to estimate calories burned.
def lcdaCalories(coords: List[List[float]] = [], BMR: dict = {}, options: dict = {}) -> dict:
    """Use the LCDA predictive model to calculate the total (and per-segment) calorie expenditure for a GPS track.

    Args:
        coords (List[List[float]]): GPS coordinates array.  Each element:
            [longitude, latitude, heading, altitude (m), accuracy (m), timestamp (ms)]
        BMR (dict): Values for calculating resting metabolic rate.
        BMR["height"] (float): Body height in cm.
        BMR["weight"] (float): Body weight in kg.
        BMR["age"] (int): Age, in years.
        BMR["sex"] = 'm'|'f' (string): Male of female.
        options (dict): Options
        options["bodyWeightKg"] (float): Body weight in kg (required).
        options["loadKg"] = 0 (float): Load/pack weight in kg (optional).
        options["waterKg"] = 0 (float): Water weight in kg carried (optional).
        options["terrain"] = 1.1 (float): Terrain coefficient (optional).  Use TERRAIN_COEFFICIENTS.
        options["smooth"] = True (Boolean): Whether to smooth GPS altitude values (optional).
        options["smoothWindow"] = 5 (int): Rolling average window size for smoothing (optional).
        options["returnSegments"] = False (Boolean): Return array of all segments calculated (optional)?

    Raises:
        ValueError: If coords array contains less than 2 items.
        ValueError: If required body weight is < 0, null, or otherwise invalid.

    Returns:
        dict: Results
        {
            totalKcal,        # Total calories burned.
            totalDistanceM,   # Total horizontal distance (meters).
            totalDurationSec, # Total elapsed time (seconds).
            avgSpeedMs,       # Average speed (m/s).
        }
    """
    if len(coords) < 2:
        raise ValueError('At least 2 coordinate points are required.')

    if not BMR or BMR["height"] <= 0 or BMR["weight"] <= 0 or BMR["age"] <= 0 or BMR["sex"] not in {'m', 'f'}:
        msg = """BMR must include the following properties:
                    \theight: positive number (cm)
                    \tweight: positive number (kg)
                    \tage: positive number (years)
                    \tsex: string \'m|f\'"""
        raise ValueError(msg)

    bodyWeightKg = options.get("bodyWeightKg", 0)
    loadKg = options.get("loadKg", 0)
    waterKg = options.get("waterKg", 0)
    terrain = options.get("terrain", 1.1)
    smooth = options.get("smooth", True)
    smoothWindow = options.get("smoothWindow", SMOOTH_DEFAULT_WINDOW)
    returnSegments = options.get("returnSegments", False)
    if not bodyWeightKg or bodyWeightKg <= 0:
        raise ValueError(f"options['bodyWeightKg'] is required and must be a positive number.")

    print('lcda parameters:')
    print(bodyWeightKg, loadKg, waterKg)
    print(terrain)
    print(smooth, smoothWindow)
    print(f'bmr {BMR}')

    track = smoothAltitude(coords, smoothWindow) if smooth else coords
    segments = []
    totalKcal = 0
    totalDistanceM = 0
    totalDurationSec = 0
    for i in range(0, len(track)):
        seg = processLcdaSegment(track[i - 1], track[i], bodyWeightKg, loadKg, waterKg, terrain, BMR)
        if seg:
            totalKcal += seg["kcal"]
            # print(f"adding seg.kcal: {seg["kcal"]} ({totalKcal})")
            totalDistanceM += seg["horizontalDistance"]
            totalDurationSec += seg["durationSec"]
            segments.append(seg)

    avgSpeedMs = totalDistanceM / totalDurationSec if totalDurationSec > 0 else 0
    results = {
        "totalKcal": totalKcal,
        "totalDistanceM": totalDistanceM,
        "totalDurationSec": totalDurationSec,
        "avgSpeedMs": avgSpeedMs
    }
    if returnSegments:
        results["segments"] = segments

    return results



In [61]:
opts = {
    "bodyWeightKg": walk_22["features"][0]["properties"]["weights"]["body"],
    "ruckWeightKg": walk_22["features"][0]["properties"]["weights"]["ruck"],
    "smooth": True,
    "smoothWindow": 5
}
BMR = {"height": 162, "weight": walk_22["features"][0]["properties"]["weights"]["body"], "age": 53, "sex": 'm'}
lcdaCalories(coords, BMR, opts)

lcda parameters:
165 0 0
1.1
True 5
bmr {'height': 162, 'weight': 165, 'age': 53, 'sex': 'm'}


{'totalKcal': 1322.0861072204539,
 'totalDistanceM': 6090.789713205702,
 'totalDurationSec': 3953,
 'avgSpeedMs': 1.5408018500393883}

In [31]:
# dir(math)

In [32]:
smooth = list()
smooth.append('one')
smooth.append('two')
for x in range(10):
    if x % 2 == 0:
        smooth.append(x)
    else:
        smooth.append(None)

print(smooth)
# print(len(smooth))
i = 0
n = len(smooth)
while i < n:
    print(i, smooth[i])
    i += 1

print(smooth[2:6+1])
newSmooth = [x for x in smooth[2:11] if x is not None and x > 1 and x < 8]
print(newSmooth)

['one', 'two', 0, None, 2, None, 4, None, 6, None, 8, None]
0 one
1 two
2 0
3 None
4 2
5 None
6 4
7 None
8 6
9 None
10 8
11 None
[0, None, 2, None, 4]
[2, 4, 6]
